In [1]:
import pandas as pd
import os

In [2]:
%load_ext autoreload
%autoreload 2

%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns

%config InlineBackend.figure_format = 'retina'
plt.rcParams['figure.figsize'] = 8, 5
plt.rcParams['font.size'] = 12
plt.rcParams['savefig.format'] = 'pdf'
# sns.set_style('dark')

In [3]:
# os.chdir('/Users/black_chick/Desktop/HSE/individual_projects/coursework-2/music-transformer')
# os.getcwd()

In [4]:
from preprocessing.processor import encode_midi, decode_midi
from preprocessing.processor import START_IDX

min_time_value = START_IDX['time_shift']
max_time_value = START_IDX['velocity'] - 1
min_note_on = START_IDX['note_on']
max_note_on = START_IDX['note_off'] - 1
min_note_off = START_IDX['note_off']
max_note_off = START_IDX['time_shift'] - 1
min_velocity = START_IDX['velocity']
max_velocity = START_IDX['end_of_scope'] - 1

In [5]:
dataset = 'bach_chorales/midi'
midi_name = '001805bw'
midi = midi_name + '.mid'
decode_dir = ''
labeling_path = 'bach_chorales/bach_choral_set_dataset.csv'

file_path = os.path.join(dataset, midi)
decode_path = os.path.join(decode_dir, midi)

In [6]:
tokens = encode_midi(file_path, drop_pauses=True)
decode_midi(tokens, decode_path)

In [7]:
labeling = pd.read_csv(os.path.join(labeling_path))
print(labeling.shape)
labeling.head(10)

(5665, 17)


,choral_ID,event_number,pitch_1,pitch_2,pitch_3,pitch_4,pitch_5,pitch_6,pitch_7,pitch_8,pitch_9,pitch_10,pitch_11,pitch_12,bass,meter,chord_label
0,000106b_,1,YES,NO,NO,NO,NO,YES,NO,NO,NO,YES,NO,NO,F,3,F_M
1,000106b_,2,YES,NO,NO,NO,YES,NO,NO,YES,NO,NO,NO,NO,E,5,C_M
2,000106b_,3,YES,NO,NO,NO,YES,NO,NO,YES,NO,NO,NO,NO,E,2,C_M
3,000106b_,4,YES,NO,NO,NO,NO,YES,NO,NO,NO,YES,NO,NO,F,3,F_M
4,000106b_,5,YES,NO,NO,NO,NO,YES,NO,NO,NO,YES,NO,NO,F,2,F_M
5,000106b_,6,NO,NO,YES,NO,NO,YES,NO,NO,NO,YES,NO,NO,D,4,D_m
6,000106b_,7,NO,NO,YES,NO,NO,YES,NO,NO,NO,YES,NO,NO,D,2,D_m
7,000106b_,8,YES,NO,NO,NO,NO,YES,NO,NO,NO,YES,NO,NO,A,3,F_M
8,000106b_,9,YES,NO,NO,NO,NO,YES,NO,NO,NO,YES,NO,NO,A,2,F_M
9,000106b_,10,NO,NO,YES,NO,NO,YES,NO,NO,NO,NO,YES,NO,Bb,5,BbM


In [8]:
labels = labeling['chord_label'].unique().tolist()

Посмотрим, для каких композиций отсутствуют MIDI

In [9]:
chorale_names = labeling['choral_ID'].unique().tolist()
not_found = [name + '.mid' for name in chorale_names if not os.path.isfile(os.path.join(dataset, name + '.mid'))]
not_found

['001805b_.mid',
 '002406bs.mid',
 '003907bv.mid',
 '012606bv.mid',
 '014505bv.mid',
 '014806bv.mid']

In [10]:
len(chorale_names)

60

Матчим аккорды в датасете с загруженным MIDI

Нюанс: в разметку не включены пустые аккорды

In [11]:
labeling = labeling[labeling['choral_ID'] == midi_name].copy()

chords = []
currently_playing = set()
prev_token = -1
for token in tokens:
    if min_time_value <= token <= max_time_value:
        if min_time_value <= prev_token <= max_time_value or not currently_playing:
            continue
        chords.append(currently_playing.copy())
    if min_note_on <= token <= max_note_on:
        currently_playing.add(token - min_note_on)
    if min_note_off <= token <= max_note_off and token - min_note_off in currently_playing:
        currently_playing.remove(token - min_note_off)
    prev_token = token

cleared = []

for chord in chords:
    notes = set()
    for note in chord:
        notes.add(note % 12 + 1)
    cleared.append(notes)
cleared

[{3, 8, 12},
 {1, 4, 8},
 {1, 3, 4, 8},
 {1, 4, 8},
 {1, 4, 6, 10},
 {3, 8, 12},
 {1, 3, 4, 8},
 {1, 4, 8},
 {1, 3, 8},
 {3, 6, 8, 12},
 {1, 4},
 {1, 4, 8},
 {1, 4, 10},
 {3, 8, 11},
 {3, 6, 10, 11},
 {1, 4, 8},
 {1, 3, 8},
 {1, 4, 8},
 {1, 4, 8, 11},
 {1, 4, 10},
 {1, 4, 8, 10},
 {3, 7, 11},
 {3, 8, 11},
 {1, 3, 8, 10},
 {1, 3, 7, 10},
 {8, 12},
 {3, 8, 12},
 {1, 4, 8},
 {1, 3, 4, 8},
 {1, 4, 8},
 {1, 4, 6, 10},
 {3, 8, 12},
 {1, 3, 4, 8},
 {1, 4, 8},
 {1, 3, 8},
 {3, 6, 8, 12},
 {1, 4},
 {1, 4, 8},
 {1, 4, 10},
 {3, 8, 11},
 {3, 6, 10, 11},
 {1, 4, 8},
 {1, 3, 8},
 {1, 4, 8},
 {1, 4, 8, 11},
 {1, 4, 10},
 {1, 4, 8, 10},
 {3, 7, 11},
 {3, 8, 11},
 {1, 3, 8, 10},
 {1, 3, 7, 10},
 {8, 12},
 {3, 8, 11},
 {3, 6, 11},
 {1, 4, 11},
 {3, 6, 11},
 {1, 6, 9, 11},
 {4, 8, 11},
 {3, 6, 8, 11},
 {1, 4, 8},
 {1, 3, 6, 9},
 {4, 8, 11},
 {4, 8, 9, 11},
 {3, 6, 11},
 {3, 6, 9, 11},
 {4, 8},
 {4, 8, 11},
 {1, 4, 8},
 {3, 8, 12},
 {3, 6, 8, 12},
 {1, 4, 8},
 {1, 4, 8, 11},
 {1, 6, 9},
 {1, 6, 8},
 {1, 

In [12]:
def readable_tokens(tokens):
    """Transforms tokens from digits to strings and digits for better readability."""
    readable = []
    for token in tokens:
        if min_time_value <= token <= max_time_value:
            readable.append(f'time_shift_{token - min_time_value + 1}')
        elif min_note_on <= token <= max_note_on:
            readable.append(f'note_on_{token - min_note_on}')
        elif min_note_off <= token <= max_note_off:
            readable.append(f'note_off_{token - min_note_off}')
        else:
            readable.append(f'velocity_{token - min_velocity}')
    return readable